# Exp 22: Face-erase test

Card: [exp22_face_erase_test.md](../../docs/neuroslope/experiments/exp22_face_erase_test.md)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from core.boxes import FACE_BOXES, Box, erasers, read_boxes
from training.config import ROOT
from training.config import VIP
from dataset.dataset import read_train
from analysis.gradcam import GradCamFigures
from analysis.plots import save_figures, use_style

N_ROWS = None  # all generic positives with face box
N_SAMPLE = 16  # exp19 sample
SEED = 0  # exp19 sample, random boxes
CONFIG_DIR = ROOT / "configs" / "models"
CONFIGS = {
    "exp16": CONFIG_DIR / "exp16_downsampling" / "resnet18.yaml",
    "exp20": CONFIG_DIR / "exp20_coarse_dropout_non_vip" / "resnet18.yaml",
    "exp21": CONFIG_DIR / "exp21_face_erase" / "resnet18.yaml",
}
CONDITIONS = ["clean", "random box", "face erased"]
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp22_face_erase_test"

# dataviz reference palette, light mode
SERIES = {"exp16": "#eb6834", "exp20": "#2a78d6", "exp21": "#1baf7a"}
use_style()

## Fold models, face boxes


In [ ]:
cams = {name: GradCamFigures(path) for name, path in CONFIGS.items()}
faces = read_boxes(FACE_BOXES)
train = read_train()
rng = np.random.default_rng(SEED)
rows, boxes = {}, {}
for cls in ["Cat", "Dog"]:
    face = {
        i: dict(faces.get(image_id, [])).get(cls)
        for i, image_id in train["image_id"].items()
    }
    generic = train.index[
        train["fold"].isin([0, 1]) & (train[cls] == 1) & ~train[VIP].any(axis=1)
    ]
    rows[cls] = [i for i in generic if face[i] is not None][:N_ROWS]
    boxes[cls] = {
        "clean": {},
        "random box": {i: Box(*face[i]).elsewhere(rng) for i in rows[cls]},
        "face erased": {i: face[i] for i in rows[cls]},
    }
    print(cls, len(rows[cls]), "rows")

## Recall: clean, random box, face erased


In [ ]:
recall = {
    (cls, name, cond): (
        cam.probs(rows[cls], cls, erasers(boxes[cls][cond])) >= cam.thresholds[cls]
    ).mean()
    for cls in ["Cat", "Dog"]
    for name, cam in cams.items()
    for cond in CONDITIONS
}
figures = {}
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)
for cls, ax in zip(["Cat", "Dog"], axes):
    for name in cams:
        y = [recall[cls, name, cond] for cond in CONDITIONS]
        ax.plot(CONDITIONS, y, marker="o", lw=2, color=SERIES[name], label=name)
        ax.annotate(
            f"{y[-1]:.3f}",
            (2, y[-1]),
            xytext=(6, 0),
            textcoords="offset points",
            va="center",
            fontsize=8,
            color=SERIES[name],
        )
    ax.set(title=f"{cls} ({len(rows[cls])} generic positives)", xlim=(-0.2, 2.5))
    ax.legend(loc="lower left", alignment="left")
axes[0].set_ylabel("recall at own best-F1 threshold")
fig.tight_layout()
figures[f"{PREFIX}_recall.png"] = fig
for key, value in recall.items():
    print(*key, f"{value:.3f}")
plt.show()

## Grad-CAM with face erased


In [ ]:
source = "exp16"  # exp19 rows drawn from exp16 OOF
for cls in ["Cat", "Dog"]:
    sample = cams[source].positive_rows(cls, N_SAMPLE, SEED)
    erased = {
        i: box
        for i in sample
        if (box := dict(faces.get(train.loc[i, "image_id"], [])).get(cls))
    }
    fig = GradCamFigures.compare(cams, cls, sample, edits=erasers(erased))
    fig.suptitle(
        f"{cls} · face erased · validation (model held image out) · {len(sample)} labeled positives",
        color="#52514e",
    )
    figures[f"{PREFIX}_grad_cam_{cls.lower()}.png"] = fig
plt.show()

In [ ]:
save_figures(figures, FIGURES)